# Multi-track voice pipeline: 270m filler + 1b speaker + qwen3.5 reasoner

All three models are mlx 4-bit and in one process, preloaded and prewarmed, with Ollama
unloaded. `bench_independent.py` measures each model alone, and this notebook compares
against it.

**Claim:** with all three tracks preloaded:
- warm median **first audio ≤ 400 ms** from end of speech (filler or backchannel)
- plain fact answers **start ≤ 1000 ms**
- routing is **≥ 80 %** correct
- contention costs the speaker **≤ 2×** its independent TTFT

**Falsifier:** any one of these fails.

Set `MULTITRACK_POLICY` to choose the policy:
- `always_filler` (A): every turn gets a filler as soon as the 270m decides.
- `route_aware` (B), the default: backchannels play at once, hold fillers play only on
  tool turns, and the reasoner is cancelled when it calls no tool.

In [1]:
import json, os, time, resource
import ollama
import tracks as T

ROUNDS = 3
POLICY = os.environ.get("MULTITRACK_POLICY", "route_aware")   # or "always_filler"
OUT = T.HERE / f"results_combined_{POLICY}.json"
unloaded = [m.model for m in ollama.ps().models]
for name in unloaded:
    ollama.generate(model=name, prompt="", keep_alive=0)
time.sleep(2)
still = [m.model for m in ollama.ps().models]
assert not still, f"Ollama still has models loaded: {still}"
items = T.load_items()
results = {"meta": {"rounds": ROUNDS, "policy": POLICY, "policy_config": T.POLICIES[POLICY], "items": [i["id"] for i in items],
                    "filler": T.FILLER_REPO, "speaker": T.SPEAKER_REPO, "reasoner": T.REASONER_REPO,
                    "ollama_unloaded": unloaded}}
results["meta"]

{'rounds': 3,
 'policy': 'always_filler',
 'policy_config': {'hold_filler': 'always',
  'cancel_reasoner_without_tool': False},
 'items': ['fact-001',
  'fact-002',
  'fact-003',
  'fact-004',
  'fact-005',
  'fact-006',
  'fact-007',
  'fact-008',
  'back-001',
  'back-002',
  'back-003',
  'back-004',
  'tool-001',
  'tool-002',
  'tool-003'],
 'filler': 'mlx-community/gemma-3-270m-it-qat-4bit',
 'speaker': 'mlx-community/gemma-3-1b-it-qat-4bit',
 'reasoner': 'mlx-community/Qwen3.5-4B-4bit',
 'ollama_unloaded': []}

## 1. Preload and prewarm everything

In [2]:
pre = {}
t = time.monotonic()
ctx = T.SharedContext()
tts = T.TTS(); pre["kokoro_load_ms"] = tts.load_ms
pre.update(tts.prewarm_and_bank())
asr = T.ASR(); pre["whisper_variant"] = asr.variant; pre["whisper_load_and_warm_ms"] = asr.prewarm()
filler = T.FillerTrack(); pre["filler_load_and_pin_ms"] = filler.load()
filler.pick("Warm up call.")
speaker = T.SpeakerTrack(ctx); pre["speaker_load_and_warm_ms"] = speaker.load()
reasoner = T.ReasonerTrack(ctx); pre["reasoner_load_and_pin_ms"] = reasoner.load()
t2 = time.monotonic(); reasoner.answer("What is two plus two?"); pre["reasoner_warm_tool_turn_ms"] = T.ms(t2)
ctx.notes.clear()
pre["total_ms"] = T.ms(t)
import mlx.core as mx
pre["mlx_active_mb_all_loaded"] = round(mx.get_active_memory() / 2**20)
results["prewarm"] = pre
{k: v for k, v in pre.items() if k not in ("bank_live_synth_ms", "bank_clip_ms")}

<venv>/lib/python3.12/site-packages/torch/nn/modules/rnn.py:1011: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)
<venv>/lib/python3.12/site-packages/torch/nn/utils/weight_norm.py:145: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


<venv>/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

{'kokoro_load_ms': 4389.7,
 'kokoro_warm_ms': 711.4,
 'bank_phrases': 11,
 'bank_speed': 1.15,
 'bank_keep_ms': 40,
 'bank_live_synth_median_ms': 167.9,
 'whisper_variant': 'accent-lora',
 'whisper_load_and_warm_ms': 12357.9,
 'filler_load_and_pin_ms': 1363.4,
 'speaker_load_and_warm_ms': 1337.8,
 'reasoner_load_and_pin_ms': 2274.9,
 'reasoner_warm_tool_turn_ms': 1150.7,
 'total_ms': 25485.6,
 'mlx_active_mb_all_loaded': 3793}

## 2. Input audio (Kokoro speaks each item once, cached in `audio/`)

In [3]:
audio = {i["id"]: T.input_audio(tts, i) for i in items}
mt = T.MultiTrack(asr, tts, filler, speaker, reasoner, ctx, policy=POLICY)
len(audio)

15

## 3. Warm rounds: one conversation per round, all 15 items

In [4]:
turns = []
for r in range(1, ROUNDS + 1):
    speaker.new_conversation()
    for item in items:
        turns.append(mt.turn(item, audio[item["id"]], round_=r))
    print(f"round {r} done")
results["turns"] = turns
results["memory"] = {"python_max_rss_mb": round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 2**20),
                     "mlx_active_mb": round(mx.get_active_memory() / 2**20),
                     "mlx_peak_mb": round(mx.get_peak_memory() / 2**20),
                     "ollama_loaded": [m.model for m in ollama.ps().models]}
results["memory"]

round 1 done


round 2 done


round 3 done


{'python_max_rss_mb': 6411,
 'mlx_active_mb': 3743,
 'mlx_peak_mb': 4577,
 'ollama_loaded': []}

## 4. Summary, contention against independent runs, verdict

In [5]:
summary = T.summarize(turns)
results["summary"] = summary
ind = json.loads((T.HERE / "results_independent.json").read_text())
contention = {
    "filler_decision_ms": (ind["filler"]["decision_ms"], summary["filler_decision_ms"]),
    "speaker_ttft_ms": (ind["speaker"]["ttft_ms_turn2plus"], summary["facts_speaker_ttft_ms"]),
    "reasoner_ttft_ms": (ind["reasoner"]["ttft_ms"], summary["reasoner_ttft_ms"]),
}
results["contention"] = {k: {"independent": a, "combined": b, "ratio": round(b / a, 2)}
                         for k, (a, b) in contention.items()}
ok, total = map(int, summary["route_accuracy"].split("/"))
checks = {
    "first_audio_le_400ms": summary["first_audio_ms"] <= 400,
    "facts_answer_start_le_1000ms": summary["facts_answer_start_ms"] <= 1000,
    "route_accuracy_ge_80pct": ok / total >= 0.8,
    "speaker_contention_le_2x": results["contention"]["speaker_ttft_ms"]["ratio"] <= 2,
}
results["verdict"] = {"checks": checks, "claim_holds": all(checks.values())}
OUT.write_text(json.dumps(results, indent=2))
print(json.dumps(summary, indent=1)); print(json.dumps(results["contention"], indent=1))
results["verdict"]

{
 "turns": 45,
 "route_accuracy": "39/45",
 "filler_fallback": 21,
 "stt_ms": 212.9,
 "filler_decision_ms": 95.4,
 "first_audio_ms": 313.3,
 "first_audio_ms_backchannel": 309.4,
 "tool_route_ms": 361.3,
 "facts_tool_routed": "6/24",
 "facts_answer_ready_ms": 764.3,
 "facts_answer_start_ms": 1041.2,
 "facts_speaker_ttft_ms": 134.0,
 "facts_correct": "21/24",
 "tools_tool_routed": "6/9",
 "tools_expected_tool_used": "6/9",
 "tools_answer_start_ms": 3173.5,
 "tools_correct": "6/9",
 "reasoner_ttft_ms": 144.8,
 "reasoner_ready_ms_tools": 1521.6
}
{
 "filler_decision_ms": {
  "independent": 18.1,
  "combined": 95.4,
  "ratio": 5.27
 },
 "speaker_ttft_ms": {
  "independent": 34.5,
  "combined": 134.0,
  "ratio": 3.88
 },
 "reasoner_ttft_ms": {
  "independent": 147.1,
  "combined": 144.8,
  "ratio": 0.98
 }
}


{'checks': {'first_audio_le_400ms': True,
  'facts_answer_start_le_1000ms': False,
  'route_accuracy_ge_80pct': True,
  'speaker_contention_le_2x': False},
 'claim_holds': False}

## 5. Round 1, turn by turn

In [6]:
for t in turns:
    if t["round"] == 1:
        tl = " | ".join(f"{c['kind']}@{c['start_ms']:.0f} {c['text']!r}" for c in t["timeline"])
        flag = "" if t["route_correct"] and t["answer_correct"] is not False else "  <-- CHECK"
        print(f"{t['id']:<9} {t['transcript']!r}\n          filler={t['filler_raw']!r} route={t['route']} tool={t.get('tool_route')} :: {tl}{flag}")

fact-001  'What is the capital of France?'
          filler='Paris.' route=hold tool=False :: filler@310 'One moment.' | answer@1035 'Paris.'
fact-002  'How many continents are there?'
          filler='Two.' route=hold tool=False :: filler@282 'One moment.' | answer@1007 'There are seven continents.'
fact-003  'What color is the sky on a clear day?'
          filler='Blue.' route=hold tool=False :: filler@331 'One moment.' | answer@1056 'Blue.'
fact-004  'who wrote Romeo and Juliet.'
          filler='Yes.' route=backchannel tool=None :: filler@302 'Yes.'  <-- CHECK
fact-005  'How many days are in a leap year?'
          filler='One day.' route=hold tool=True :: filler@328 'One moment.' | deferred@2007 '366.'
fact-006  'What is the largest ocean on Earth?'
          filler='The largest island in the world.' route=hold tool=False :: filler@362 'One moment.' | answer@1087 'The Pacific Ocean.'
fact-007  'What gas do plants absorb from the air?'
          filler='Carbon dioxide.' route=ho

## 6. Check: summary recomputes from results_combined.json

In [7]:
saved = json.loads(OUT.read_text())
assert T.summarize(saved["turns"]) == saved["summary"]
assert len(saved["turns"]) == ROUNDS * len(items)
f"{OUT.name} consistent" 

'results_combined_always_filler.json consistent'

## 7. Listen (round 1: a fact, a remark, a tool question)

In [8]:
from IPython.display import Audio, display
for t in turns:
    if t["round"] == 1 and t["id"] in ("fact-001", "back-002", "tool-001"):
        print(t["id"], [c["text"] for c in t["timeline"]])
        display(Audio(str(T.HERE / t["reply_wav"])))

fact-001 ['One moment.', 'Paris.']


back-002 ['Okay.']


tool-001 ['Good question.', '17 * 23 = 391.']
